In [ ]:
!pip install gurobipy>=10
from gurobipy import Model, GRB, quicksum
import gurobipy as gb
# Create an environment with WLS license
params = {
"WLSACCESSID": "86a97016-e4e4-4614-9a0c-c845ff16567b",
"WLSSECRET": "135f418c-de65-4e2f-bbd5-c3dd79fd70f9",
"LICENSEID": 2503807
}
env = gb.Env(params=params)


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2503807
Academic license 2503807 - for non-commercial use only - registered to te___@hi.is


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd

Start by using generated data, then we know the true underlying distributions. Let $\xi $ be $N(0,1)$ and set $y= c_0 + c_1\xi + \epsilon$  where $\epsilon$ is an error term s.t.
$$
\epsilon = \begin{cases} N(0,\sigma^2) \text{ with probability } 1-p \\
N(0,\tau^2) \text{ with probability } p
\end{cases}
$$
where $\sigma^2 << \tau^2$ and p is small. Most of the time we have little noise but occasionally get large errors.

Also test asymmetric error: make the large deviations state dependent and asymetric - creates regions where large errors are more likely.  Makes sense only if decision maker wants to avoid large losses in specific parts of the covariate space

Let $\xi $ be $N(0,1)$ or Uniform(-2,2) and set $y= c_0 + c_1\xi + \epsilon$  where $\epsilon$ is an error term s.t.
$$
\epsilon = \begin{cases} N(0,1) \text{ if } \xi < 1 \\
N(0,1) \text{ with prob } 1-p \text{ and } N(\mu,\tau^2) \text{ with prob } p \text{ if } \xi \geq 1
\end{cases}
$$
where $\mu $ and $\tau^2$ are large and p is small.


In [ ]:
# Genarate data
def simulate_data(n, b0 = 1, b1=2, prob = 0.05, sigma = 1, tau =10, seed = None):
  rng = np.random.default_rng(seed)
  xi = rng.normal(loc = 0.0, scale = 1.0, size = n)
  u = rng.uniform(0,1,n)
  e_small = rng.normal(loc = 0.0, scale = sigma, size = n)
  e_large = rng.normal(loc = 0.0, scale = tau, size = n)
  epsilon = np.where(u < prob, e_large, e_small)
  y = b0 + b1*xi + epsilon
  return xi,y

def simulate_asymmetric(n, b0 = 1.0, b1 = 2.0, prob = 0.2, sigma = 1.0, mu_large = 6.0,
                        tau_large = 2.0, trigger = 0.5, seed = None):
  rng = np.random.default_rng(seed)
  xi = rng.uniform(-2.0,2.0, size = n)
  u = rng.uniform(0.0, 1.0, size = n)

  e_small = rng.normal(0.0, sigma, size = n)
  e_large = rng.normal(mu_large, tau_large, size = n)

  epsilon = np.where((xi > trigger) & (u < prob), e_large, e_small)
  y = b0 + b1*xi + epsilon
  return xi,y

# Benchmark coefficients (OLS)
def fit_ols(xi,y):
  x = sm.add_constant(xi)
  ols_fit = sm.OLS(y,x).fit()
  coeff_ols = ols_fit.params
  return coeff_ols


The FSD optimization model for linear class

In [ ]:
def model(xi, y):
  S = len(y)
  c_ols = fit_ols(xi,y)
  Lb = np.abs(y - (c_ols[0] + c_ols[1]*xi))

  t, counts = np.unique(Lb, return_counts = True) # ordered support points
  q = counts/S # prob of each scenario for benchmark
  K = len(t)

  p = np.full(S, 1/S) # prob of each scenario of xi

  # Model
  m = Model(env = env)

  # bounds on c
  C = 20
  # variables
  c0 = m.addVar(lb = -C, ub = C, name = "c0")
  c1 = m.addVar(lb = -C, ub = C, name = "c1")

  beta = m.addVars(S, K , vtype = GRB.BINARY, name = "beta")

  # linearize absolute value
  r = m.addVars(S, lb = -GRB.INFINITY, name = "r")
  u_abs = m.addVars(S, lb = 0.0, name = "u_abs")

  y_max = float(np.max(np.abs(y)))
  xi_max = float(np.max(np.abs(xi)))

  M = np.maximum(0.0, y_max + (C + C*xi_max) - t) # get upper bound with triangle ineq

  # objective
  m.setObjective(1/S*quicksum(u_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - (c0 + c1*xi[s]) for s in range(S))
  m.addConstrs((u_abs[s] >= r[s] for s in range(S)))
  m.addConstrs((u_abs[s] >= -r[s] for s in range(S)))

  prob_bench = 1.0 - np.cumsum(q)
  m.addConstrs(quicksum(p[s]*beta[s,k] for s in range(S)) <= prob_bench[k] for k in range(K))
  m.addConstrs(M[k]*beta[s,k] >= u_abs[s] - t[k] for k in range(K) for s in range(S))

  # Solve - add extra tolerance
  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.optimize()
  return c0.X, c1.X


In [ ]:
n = 100
xi, y = simulate_data(n=n, seed = 1234)
# xi, y = simulate_asymmetric(n)
c0, c1 = model(xi,y)

c_fsd = np.array([c0, c1])
c_ols = fit_ols(xi,y)

# LAD
def fit_lad(xi,y):
  X = sm.add_constant(xi)
  qr = sm.QuantReg(y, X).fit(q = 0.5,  max_iter=10000)
  return np.asarray(qr.params)

c_lad = fit_lad(xi,y)

print("LAD: ", c_lad)
print("OLS: ", c_ols)
print("FSD: ", c_fsd)

Need to check if solution is valid as this is MILP

In [ ]:
def check_fsd_sol(xi, y, c_fsd, c_ols):
  L_fsd = np.abs(y- (c_fsd[0] + c_fsd[1]*xi))
  L_ols = np.abs(y - (c_ols[0] + c_ols[1]*xi))

  n = len(L_ols)
  step_size = 1/n

  #benchmark
  t, counts = np.unique(L_ols, return_counts = True)
  q = counts/n

  tol = 1e-6

  cdf_fsd = np.array([(L_fsd <= tk + tol).mean() for tk in t])
  cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])

  gaps_cdf = cdf_fsd - cdf_ols # ols cdf should be under ols always

  min_gap = gaps_cdf.min()
  #print("cdf gap", min_gap)
  #print("FSD holds:", min_gap >= -tol)

  return min_gap >= -tol

check = check_fsd_sol(xi = xi, y = y, c_fsd = c_fsd, c_ols = c_ols)


FSD Monte carlo - linear model

In [ ]:
def run_mc(n_rep , n):
  mean_fsd = np.zeros((n_rep, 2))
  c_ols = np.zeros((n_rep,2))
  c_lad = np.zeros((n_rep,2))
  c_fsd = np.zeros((n_rep,2))
  mae_fsd = np.zeros((n_rep))
  mae_ols = np.zeros((n_rep))
  mae_lad = np.zeros((n_rep))
  coef_close = np.zeros(n_rep)
  dom_ols = np.zeros(n_rep) # 0 if not dominated
  for i in range(n_rep):
    if (i % 10 == 0):
      print("iteration:", i)
    seed = 1234 + i # for reproducibility
    xi, y = simulate_data(n, seed = seed)
    # xi, y = simulate_asymmetric(n, seed = seed)
    c_fsd[i] = model(xi, y)
    c_lad[i] = fit_lad(xi,y)
    c_ols[i] = fit_ols(xi, y)

    mae_fsd[i] = np.mean(abs(y - (c_fsd[i,0] + c_fsd[i,1] * xi)))
    mae_ols[i] = np.mean(abs(y - (c_ols[i,0] + c_ols[i,1] * xi)))
    mae_lad[i] = np.mean(abs(y - (c_lad[i,0] + c_lad[i,1] * xi)))

    if (check_fsd_sol(xi, y, c_fsd[i], c_ols[i])):
      L_ols = np.abs(y - (c_ols[i,0] + c_ols[i,1]*xi))
      L_lad = np.abs(y - (c_lad[i,0] + c_lad[i,1]*xi))
      t, counts = np.unique(L_ols, return_counts = True)

      cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])
      cdf_lad = np.array([(L_lad <= tk).mean() for tk in t])
      if (np.min(cdf_lad - cdf_ols)>= -1e-8):
        dom_ols[i] = 1

      if (np.linalg.norm(c_fsd[i] - c_ols[i], ord=np.inf) <= 1e-7):
        coef_close[i] = 1
      else:
        print("ols", c_ols[i])
        print("fsd", c_fsd[i])

    else:
      print("fsd failed at iteration", i)

  print("Average MAE (OLS): ", mae_ols.mean())
  print("Average MAE (LAD): ", mae_lad.mean())
  print("Average MAE (FSD): ", mae_fsd.mean())
  print("Average coef (FSD): ", c_fsd.mean(axis=0))
  print("Average coef (OLS): ", c_ols.mean(axis = 0))
  print("Average coef (LAD): ", c_lad.mean(axis = 0))
  print("Fraction where OLS = FSD", coef_close.mean())
  print("Fraction where L_LAD dominates L_OLS", dom_ols.mean())


run_mc(n_rep = 500, n = 100)

FSD Multidimensional model

In [ ]:
def simulate_multid_data(n, dim, beta, b0 = 1.0, prob = 0.05, sigma = 1, tau = 10, seed = None):
  rng = np.random.default_rng(seed)
  xi = rng.normal(loc = 0.0, scale = 1.0, size=(n,dim))
  beta = np.asarray(beta)
  u = rng.uniform(0,1,n)
  e_small = rng.normal(loc = 0.0, scale = sigma, size = n)
  e_large = rng.normal(loc = 0.0, scale = tau, size = n)
  epsilon = np.where(u < prob, e_large, e_small)
  y = b0 + xi@beta + epsilon

  return xi, y

def simulate_multid_asymmetric(n, dim, beta, b0 = 1.0, prob = 0.2, sigma = 1.0, mu_large = 6.0,
                        tau_large = 2.0, trigger = 0.5, seed = None):
  rng = np.random.default_rng(seed)
  xi = rng.uniform(-2.0,2.0, size = (n,dim))
  beta = np.asarray(beta)

  u = rng.uniform(0.0, 1.0, size = n)

  e_small = rng.normal(0.0, sigma, size = n)
  e_large = rng.normal(mu_large, tau_large, size = n)

  epsilon = np.where((xi > trigger) & (u < prob), e_large, e_small)
  y = b0 + xi@beta + epsilon
  return xi,y

def fsd_multi(xi,y):
  S, d = xi.shape
  X = sm.add_constant(xi)
  c_ols = fit_ols(xi,y)

  Lb = np.abs(y - X@c_ols)
  t, counts = np.unique(Lb, return_counts = True)
  q = counts/S
  K = len(t)
  p = np.full(S, 1/S)

  # Model
  m = Model(env = env)
  # upper bound
  C = 20

  # variables
  # c[0] intercept and c[1],..,c[d] slopes
  c = m.addVars(d+1, lb = -C, ub = C, name = "c")
  beta = m.addVars(S, K , vtype = GRB.BINARY, name = "beta")
  r = m.addVars(S,lb = -GRB.INFINITY, name = "r")
  u_abs = m.addVars(S, lb = 0.0, name = "u_abs")

  # Need upper bound on u_abs[s] = |y[s] - X[s] @ c| use triangle ineq. get
  # |y[s] - X[s]@c| <= |y[s]| + |X[s]@c|<= max|y| + sum_j max|X_j| * max|c_j|
  y_max = float(np.max(np.abs(y)))
  X_max = np.max(np.abs(X), axis = 0)
  M = np.maximum(0.0, y_max + C*np.sum(X_max) - t)

  # objective
  m.setObjective(1/S*quicksum(u_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - quicksum(c[j]*X[s,j] for j in range(d+1)) for s in range(S))
  m.addConstrs(u_abs[s] == gb.abs_(r[s]) for s in range(S))

  prob_bench = 1.0 - np.cumsum(q)
  m.addConstrs(quicksum(p[s]*beta[s,k] for s in range(S)) <= prob_bench[k] for k in range(K))
  m.addConstrs(M[k]*beta[s,k] >= u_abs[s] - t[k] for k in range(K) for s in range(S))

  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.optimize()

  return np.array([c[j].X for j in range(d + 1)])

In [ ]:
def check_fsd_sol_multi(xi, y, c_fsd, c_ols, tol=1e-6):
    X = sm.add_constant(xi)

    L_fsd = np.abs(y - X@c_fsd)
    L_ols = np.abs(y - X @ c_ols)

    t = np.unique(L_ols)

    cdf_fsd = np.array([(L_fsd <= tk + tol).mean() for tk in t])
    cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])

    min_gap = np.min(cdf_fsd - cdf_ols)

    return min_gap >= -tol

def run_mc_multid(dim, beta, n_rep = 100, n = 100):
  mean_fsd = np.zeros((n_rep, dim+1))
  c_ols = np.zeros((n_rep,dim+1))
  c_lad = np.zeros((n_rep, dim + 1))
  c_fsd = np.zeros((n_rep, dim + 1))
  coef_close = np.zeros(n_rep)
  dom_ols = np.zeros(n_rep) # 0 if not dominated
  for i in range(n_rep):
    if (i % 10 == 0):
      print("iteration:", i)
    seed = 1234 + i # for reproducibility
    xi, y = simulate_multid_data(n, dim = dim, beta = beta, seed = seed) # or asymetric data
    c_fsd[i] = fsd_multi(xi, y)
    c_lad[i] = fit_lad(xi,y)
    c_ols[i] = fit_ols(xi, y)


    if (check_fsd_sol_multi(xi, y, c_fsd[i], c_ols[i])):
      X = sm.add_constant(xi)

      L_ols = np.abs(y - X@c_ols[i])
      L_lad = np.abs(y - X @c_lad[i])
      t, counts = np.unique(L_ols, return_counts = True)

      cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])
      cdf_lad = np.array([(L_lad <= tk).mean() for tk in t])
      if (np.min(cdf_lad - cdf_ols)>= -1e-8):
        dom_ols[i] = 1

      if (np.linalg.norm(c_fsd[i] - c_ols[i], ord=np.inf) <= 1e-7):
        coef_close[i] = 1
      else:
        print("ols", c_ols[i])
        print("fsd", c_fsd[i])

    else:
      print("fsd failed at iteration", i)

  print("Average coef (FSD): ", c_fsd.mean(axis=0))
  print("Average coef (OLS): ", c_ols.mean(axis = 0))
  print("Average coef (LAD): ", c_lad.mean(axis = 0))
  print("Fraction where OLS = FSD", coef_close.mean())
  print("Fraction where L_LAD dominates L_OLS", dom_ols.mean())


run_mc_multid(2, beta = [2.0, 3.0], n_rep = 100, n = 100)

Enforce strict inequality at at least one treshold for FSD

In [ ]:
def strict_fsd(xi, y):
  S = len(y)
  c_ols = fit_ols(xi,y)
  Lb = np.abs(y - (c_ols[0] + c_ols[1]*xi))

  t, counts = np.unique(Lb, return_counts = True) # ordered support points
  q = counts/S # prob of each scenario for benchmark
  K = len(t)

  p = np.full(S, 1/S) # prob of each scenario of xi

  # Model
  m = Model(env = env)

  # bounds on c
  C0 = 20
  C1 = 20
  # variables
  c0 = m.addVar(lb = -C0, ub = C0, name = "c0")
  c1 = m.addVar(lb = -C1, ub = C1, name = "c1")

  beta = m.addVars(S, K , vtype = GRB.BINARY, name = "beta")

  # linearize absolute value
  r = m.addVars(S, lb = -GRB.INFINITY, name = "r")
  u_abs = m.addVars(S, lb = 0.0, name = "u_abs")
  z = m.addVars(K, vtype = GRB.BINARY, name = "z")

  y_max = float(np.max(np.abs(y)))
  xi_max = float(np.max(np.abs(xi)))

  M = np.maximum(0.0, y_max + (C0 + C1*xi_max) - t) # get upper bound with triangle ineq

  # objective
  m.setObjective(1/S*quicksum(u_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - (c0 + c1*xi[s]) for s in range(S))
  m.addConstrs((u_abs[s] >= r[s] for s in range(S)))
  m.addConstrs((u_abs[s] >= -r[s] for s in range(S)))

  eps = 1.0/S
  prob_bench = 1.0 - np.cumsum(q)
  m.addConstrs(quicksum(p[s]*beta[s,k] for s in range(S)) <= prob_bench[k] - eps*z[k] for k in range(K))
  m.addConstrs(M[k]*beta[s,k] >= u_abs[s] - t[k] for k in range(K) for s in range(S))
  m.addConstr(quicksum(z[k] for k in range(K))>= 1)


  # Solve
  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.optimize()
  if m.status != GRB.OPTIMAL:
    #print("Failed, status:", m.status)
    return None

  return np.array([c0.X, c1.X])


In [ ]:
def check_strict_fsd_sol(xi, y, c_fsd, c_ols):
    L_fsd = np.abs(y - (c_fsd[0] + c_fsd[1] * xi))
    L_ols = np.abs(y - (c_ols[0] + c_ols[1] * xi))

    S = len(y)
    eps = 1 / S

    t = np.unique(L_ols)
    tol=1e-6
    cdf_fsd = np.array([(L_fsd <= tk+tol).mean() for tk in t])
    cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])

    gaps = cdf_fsd - cdf_ols
    weak_check = gaps.min() >= -tol
    strict_check = gaps.max() >= eps - tol

    return weak_check and strict_check # both need to hold



def run_mc_strict(n_rep = 100, n = 100):
  solved = np.zeros(n_rep)
  c_ols = np.zeros((n_rep,2))
  c_lad = np.zeros((n_rep, 2))
  c_fsd = np.zeros((n_rep, 2))
  coef_close = np.zeros(n_rep)
  dom_ols = np.zeros(n_rep) # 0 if not dominated
  for i in range(n_rep):
    if (i % 10 == 0):
      print("iteration:", i)
    seed = 1234 + i # for reproducibility
    xi, y = simulate_asymmetric(n, seed = seed) # or simulate_data
    sol = strict_fsd(xi, y)
    c_lad[i] = fit_lad(xi,y)
    c_ols[i] = fit_ols(xi, y)

    if (sol is not None):
      solved[i] = 1
      c_fsd[i] = sol
      print("FSD:", c_fsd[i])
      print("OLS:", c_ols[i])
      print("Difference", np.linalg.norm(c_fsd[i] - c_ols[i], ord=np.inf))

      if (check_strict_fsd_sol(xi, y, c_fsd[i], c_ols[i])):
        L_ols = np.abs(y - (c_ols[i,0] + c_ols[i,1]*xi))
        L_lad = np.abs(y - (c_lad[i,0] + c_lad[i,1]*xi))
        t, counts = np.unique(L_ols, return_counts = True)

        cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])
        cdf_lad = np.array([(L_lad <= tk).mean() for tk in t])
        if (np.min(cdf_lad - cdf_ols)>= -1e-8):
          dom_ols[i] = 1

        if (np.linalg.norm(c_fsd[i] - c_ols[i], ord=np.inf) <= 1e-7):
          coef_close[i] = 1
        else:
          print("ols", c_ols[i])
          print("fsd", c_fsd[i])

      else:
        print("fsd failed at iteration", i)

  print("Fraction of valid solutions", solved.mean())

run_mc_strict(n_rep = 100, n = 100)

FSD Piecewise linear model:
Let $\xi $ be $Unif(-2,2)$ and set $y= c_0 + c_1\xi + c_2(\xi - a)_+ + \epsilon$, where $\epsilon$ is an error term s.t.
$$
\epsilon = \begin{cases} N(0,1) \text{ if } \xi < 1 \\
N(0,1) \text{ with prob } 1-p \text{ and } N(\mu,\tau^2) \text{ with prob } p \text{ if } \xi \geq 1
\end{cases}
$$
where $\sigma^2 << \tau^2$ and p is small, a = 0.

In [ ]:
def simulate_data_piecewise(n, b0 = 1, b1 = 2, b2 = 3, prob = 0.2, sigma = 1.0, mu = 12.0, tau = 2,
                            kappa = 0.0, trigger = 1.0, seed = None):
  rng = np.random.default_rng(seed)

  xi = rng.uniform(-2.0, 2.0, size = n)
  z = np.maximum(xi - kappa, 0)

  u = rng.uniform(0.0, 1.0, size = n)
  e_small = rng.normal(0.0, sigma, size = n)
  e_large = rng.normal(mu, tau, size = n)

  epsilon = np.where((xi > trigger) & (u < prob), e_large, e_small)

  y = b0 + b1*xi + b2*z + epsilon
  return xi,y

def fit_ols_piecewise(xi, y, kappa = 0.0):
  intercept = np.ones(len(xi))
  z = np.maximum(xi - kappa, 0.0)

  X = np.column_stack([intercept, xi, z])
  fit = sm.OLS(y,X).fit()
  return np.asarray(fit.params)

def fit_lad_piecewise(xi, y, kappa = 0.0):
  z = np.maximum(xi - kappa, 0.0)
  X = np.column_stack([np.ones(len(xi)), xi, z])
  fit = sm.QuantReg(y,X).fit(q=0.5, max_iter=10000)
  return np.asarray(fit.params)

def piecewise_fsd(xi, y, kappa = 0.0, time_limit = 120):
  S = len(y)
  z = np.maximum(xi - kappa, 0.0)
  # benchmark
  c_ols = fit_ols(xi,y)
  Lb = np.abs(y-(c_ols[0] + c_ols[1]*xi))
  t, counts = np.unique(Lb, return_counts = True) # ordered support points
  q = counts/S # prob of each scenario for benchmark
  K = len(t)
  p = np.full(S, 1.0/S) # prob of each scenario of xi

  # Model
  m = Model(env = env)

  # bounds on c
  C = 20
  # variables
  c0 = m.addVar(lb = -C, ub = C, name = "c0")
  c1 = m.addVar(lb = -C, ub = C, name = "c1")
  c2 = m.addVar(lb = -C, ub = C, name = "c2")

  beta = m.addVars(S, K , vtype = GRB.BINARY, name = "beta")

  # linearize absolute value
  r = m.addVars(S, lb = -GRB.INFINITY, name = "r")
  u_abs = m.addVars(S, lb = 0.0, name = "u_abs")

  y_max = float(np.max(np.abs(y)))
  xi_max = float(np.max(np.abs(xi)))
  z_max = float(np.max(np.abs(z)))

  M = np.maximum(0.0, y_max + C + C*xi_max + C*z_max - t) # get upper bound with triangle ineq

  # objective
  m.setObjective(1/S*quicksum(u_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - (c0 + c1*xi[s] + c2*z[s]) for s in range(S))
  m.addConstrs((u_abs[s] >= r[s] for s in range(S)))
  m.addConstrs((u_abs[s] >= -r[s] for s in range(S)))

  prob_bench = 1.0 - np.cumsum(q)
  m.addConstrs(quicksum(p[s]*beta[s,k] for s in range(S)) <= prob_bench[k] for k in range(K))
  m.addConstrs(M[k]*beta[s,k] >= u_abs[s] - t[k] for k in range(K) for s in range(S))

  # Solve
  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.setParam("TimeLimit", time_limit)

  m.optimize()

  if m.status != GRB.OPTIMAL:
    #print("failed. Status:", m.status)
    #print("Solution count:", m.SolCount)
    return None, m.status
  return np.array([c0.X, c1.X, c2.X]), m.status

def check_fsd_sol_piecewise(xi, y, kappa, c_fsd, c_ols):
  z = np.maximum(0.0, xi - kappa)
  L_fsd = np.abs(y- (c_fsd[0] + c_fsd[1]*xi + c_fsd[2]*z))
  L_ols = np.abs(y - (c_ols[0] + c_ols[1]*xi))

  n = len(L_ols)

  #benchmark
  t, counts = np.unique(L_ols, return_counts = True)

  tol = 1e-6

  cdf_fsd = np.array([(L_fsd <= tk + tol).mean() for tk in t])
  cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])

  gaps_cdf = cdf_fsd - cdf_ols # ols cdf should be under ols always

  min_gap = gaps_cdf.min()

  return min_gap >= -tol


In [ ]:
def run_mc_piecewise(n_rep, n = 100, kappa = 0.0):
  solved = np.zeros(n_rep, dtype = bool)
  c_ols = np.zeros((n_rep,2))
  c_piece_lad = np.zeros((n_rep,3))
  c_piece_ols = np.zeros((n_rep,3))
  c_fsd = np.zeros((n_rep,3))
  mae_fsd = np.zeros((n_rep))
  mae_piece_ols = np.zeros((n_rep))
  mae_piece_lad = np.zeros((n_rep))
  mae_lin_ols = np.zeros((n_rep))

  q95_loss_fsd = np.zeros((n_rep))
  q95_loss_piece_lad = np.zeros((n_rep))
  q95_loss_piece_ols = np.zeros((n_rep))
  q95_loss_ols = np.zeros((n_rep))

  q99_loss_fsd = np.zeros((n_rep))
  q99_loss_piece_lad = np.zeros((n_rep))
  q99_loss_piece_ols = np.zeros((n_rep))
  q99_loss_ols = np.zeros((n_rep))

  piece_olsVSfsd_mae = np.zeros((n_rep))
  piece_olsVSfsd_95 = np.zeros((n_rep))
  piece_olsVSfsd_99 = np.zeros((n_rep))

  piece_ladVSfsd_mae = np.zeros((n_rep))
  piece_ladVSfsd_95 = np.zeros((n_rep))
  piece_ladVSfsd_99 = np.zeros((n_rep))

  coef_close_ols = np.zeros(n_rep)
  coef_close_lad = np.zeros(n_rep)

  for i in range(n_rep):
    if (i % 10 == 0):
      print("iteration:", i)
    seed = 542 + i # for reproducibility
    xi, y = simulate_data_piecewise(n, seed = seed, kappa = kappa)
    z = np.maximum(xi - kappa, 0.0)

    sol,status = piecewise_fsd(xi, y, kappa = kappa)
    if (status != GRB.OPTIMAL):
      print("FSD not solved at iteration", i, "status:", status)
      continue # if not solved optimally go to next dataset
    c_fsd[i] = sol
    c_piece_lad[i] = fit_lad_piecewise(xi,y, kappa = kappa)
    c_ols[i] = fit_ols(xi, y)
    c_piece_ols[i] = fit_ols_piecewise(xi, y, kappa = kappa)

    mae_fsd[i] = np.mean(abs(y - (c_fsd[i,0] + c_fsd[i,1] * xi + c_fsd[i,2]*z )))
    mae_lin_ols[i] = np.mean(abs(y - (c_ols[i,0] + c_ols[i,1] * xi)))
    mae_piece_lad[i] = np.mean(abs(y - (c_piece_lad[i,0] + c_piece_lad[i,1] * xi + c_piece_lad[i,2]*z)))
    mae_piece_ols[i] = np.mean(abs(y - (c_piece_ols[i,0] + c_piece_ols[i,1] * xi + c_piece_ols[i,2]*z)))

    if (check_fsd_sol_piecewise(xi, y, kappa, c_fsd[i], c_ols[i] )):
      solved[i] = True
      if(mae_fsd[i] < mae_piece_ols[i] - 1e-6):
        piece_olsVSfsd_mae[i] = 1
      if(mae_fsd[i]< mae_piece_lad[i] -1e-6):
        piece_ladVSfsd_mae[i] = 1

      L_lin_ols = np.abs(y - (c_ols[i,0] + c_ols[i,1]*xi))
      L_piece_ols = np.abs(y - (c_piece_ols[i,0] + c_piece_ols[i,1]*xi + c_piece_ols[i,2]*z))
      L_piece_lad = np.abs(y - (c_piece_lad[i,0] + c_piece_lad[i,1]*xi + c_piece_lad[i,2]*z))
      L_fsd = np.abs(y - (c_fsd[i,0] + c_fsd[i,1]*xi + c_fsd[i,2]*z))

      q95_loss_fsd[i] = float(np.quantile(L_fsd, 0.95))
      q95_loss_piece_ols[i] = float(np.quantile(L_piece_ols, 0.95))
      q95_loss_piece_lad[i] = float(np.quantile(L_piece_lad, 0.95))
      q95_loss_ols[i] = float(np.quantile(L_lin_ols, 0.95))

      q99_loss_fsd[i] = float(np.quantile(L_fsd, 0.99))
      q99_loss_piece_ols[i] = float(np.quantile(L_piece_ols, 0.99))
      q99_loss_piece_lad[i] = float(np.quantile(L_piece_lad, 0.99))
      q99_loss_ols[i] = float(np.quantile(L_lin_ols, 0.99))

      if(q95_loss_fsd[i] <q95_loss_piece_ols[i]-1e-6):
        piece_olsVSfsd_95[i] = 1
      if(q95_loss_fsd[i]< q95_loss_piece_lad[i] -1e-6):
        piece_ladVSfsd_95[i] = 1

      if(q99_loss_fsd[i] < q99_loss_piece_ols[i]-1e-6):
        piece_olsVSfsd_99[i] = 1
      if(q99_loss_fsd[i] < q99_loss_piece_lad[i]-1e-6):
        piece_ladVSfsd_99[i] = 1

      if (np.linalg.norm(c_fsd[i] - c_piece_ols[i], ord=np.inf) <= 1e-6):
        coef_close_ols[i] = 1
      #else:
      #  print("piecewise ols", c_piece_ols[i])
      #  print("fsd", c_fsd[i])
      if (np.linalg.norm(c_fsd[i] - c_piece_lad[i], ord=np.inf) <= 1e-6):
        coef_close_lad[i] = 1
      #else:
      #  print("piecewise lad", c_piece_lad[i])
      #  print("fsd", c_fsd[i])

    else:
      print("fsd failed at iteration", i)
  print("Number of solved and valid instances", solved.sum())

  print("Average MAE (OLS): ", mae_lin_ols[solved].mean()) # only calculate solved instances
  print("Average MAE (piecewise LAD): ", mae_piece_lad[solved].mean())
  print("Average MAE (FSD): ", mae_fsd[solved].mean())
  print("Average MAE (piecewise OLS): ", mae_piece_ols[solved].mean())

  print("Average q95 loss (linear OLS):", q95_loss_ols[solved].mean())
  print("Average q95 loss (piecewise OLS):", q95_loss_piece_ols[solved].mean())
  print("Average q95 loss (piecewise LAD):", q95_loss_piece_lad[solved].mean())
  print("Average q95 loss (FSD):", q95_loss_fsd[solved].mean())

  print("Average q99 loss (linear OLS):", q99_loss_ols[solved].mean())
  print("Average q99 loss (piecewise OLS):", q99_loss_piece_ols[solved].mean())
  print("Average q99 loss (piecewise LAD):", q99_loss_piece_lad[solved].mean())
  print("Average q99 loss (FSD):", q99_loss_fsd[solved].mean())

  print("Fraction where FSD beats piecewise OLS in MAE", f"{piece_olsVSfsd_mae[solved].mean():.2f}")
  print("Fraction where FSD beats piecewise OLS in q95", f"{piece_olsVSfsd_95[solved].mean():.2f}")
  print("Fraction where FSD beats piecewise OLS in q99", f"{piece_olsVSfsd_99[solved].mean():.2f}")
  print("Fraction where FSD beats piecewise LAD in MAE", f"{piece_ladVSfsd_mae[solved].mean():.2f}")
  print("Fraction where FSD beats piecewise LAD in q95", f"{piece_ladVSfsd_95[solved].mean():.2f}")
  print("Fraction where FSD beats piecewise LAD in q99", f"{piece_ladVSfsd_99[solved].mean():.2f}")

  print("Fraction where FSD = piecewise OLS (tol 10^(-6))", coef_close_ols[solved].mean())
  print("Fraction where FSD = piecewise LAD (tol 10^(-6))", coef_close_lad[solved].mean())

  '''
  # look better into the differences
  diff_q99_lad = q99_loss_fsd[solved] - q99_loss_piece_lad[solved]
  print("Quantiles of q99 difference FSD - LAD:",
        np.quantile(diff_q99_lad, [0.05, 0.25, 0.5, 0.75, 0.95]))
  print("Largest values of q99_FSD - q99_LAD:", np.sort(diff_q99_lad)[-10:])
  print("Largest values of q99_LAD - q99_FSD:",np.sort(-diff_q99_lad)[-10:])

  diff_q95_lad = q95_loss_fsd[solved] - q95_loss_piece_lad[solved]
  print("Quantiles of q95 difference FSD - LAD:", np.quantile(diff_q95_lad, [0.05, 0.25, 0.5, 0.75, 0.95]))
  print("Largest values of q95_FSD - q95_LAD:", np.sort(diff_q95_lad)[-10:])
  print("Largest values of q95_LAD - q95_FSD:",np.sort(-diff_q95_lad)[-10:])

  diff_q95_ols = q95_loss_fsd[solved] - q95_loss_piece_ols[solved]
  print("Quantiles of q95 difference FSD - OLS:", np.quantile(diff_q95_ols, [0.05, 0.25, 0.5, 0.75, 0.95]))
  print("Largest values of q95_FSD - q95_OLS:", np.sort(diff_q95_ols)[-10:])
  print("Largest values of q95_OLS - q95_FSD:",np.sort(-diff_q95_ols)[-10:])

  diff_q99_ols = q99_loss_fsd[solved] - q99_loss_piece_ols[solved]
  print("Quantiles of q99 difference FSD - OLS:", np.quantile(diff_q99_ols, [0.05, 0.25, 0.5, 0.75, 0.95]))
  print("Largest values of q99_FSD - q99_OLS:", np.sort(diff_q99_ols)[-10:])
  print("Largest values of q99_OLS - q99_FSD:",np.sort(-diff_q99_ols)[-10:])

  diff_ols_mae = mae_fsd[solved] - mae_piece_ols[solved]
  print("Quantiles of MAE difference FSD - OLS:", np.quantile(diff_ols_mae, [0.05, 0.25, 0.5, 0.75, 0.95]))
  print("Largest values of MAE_FSD - MAE_OLS:", np.sort(diff_ols_mae)[-10:])
  print("Largest values of MAE_OLS - MAE_FSD:",np.sort(-diff_ols_mae)[-10:])
  '''

run_mc_piecewise(n_rep = 500, n = 100, kappa = 0.0)

FSD Quadratic model

 $y = b_0 +b_1\boldsymbol{\xi} + b_2 \boldsymbol{\xi}^2 + \epsilon$

In [ ]:
def simulate_data_quadratic(n,b0 = 1.0, b1 = 0.0, b2 = 5.0, prob = 0.2, sigma = 1.0, mu = 5.0, tau = 2.0, trigger = 1.0, seed = None):

  rng = np.random.default_rng(seed)

  xi = rng.uniform(-2.0, 2.0, size = n)

  u = rng.uniform(0.0, 1.0, size = n)
  e_small = rng.normal(0.0, sigma, size = n)
  e_large = rng.normal(mu, tau, size = n)

  epsilon = np.where((xi > trigger) & (u < prob), e_large, e_small)

  y = b0 + b1*xi + b2*xi**2 + epsilon
  return xi,y
'''
# Genarate data
def simulate_data_quadratic(n, b0 = 1, b1=2, b2 = 2, prob = 0.05, sigma = 1, tau =10, seed = None):
  rng = np.random.default_rng(seed)
  xi = rng.uniform(-2.0,2.0, size = n)
  u = rng.uniform(0,1,n)
  e_small = rng.normal(loc = 0.0, scale = sigma, size = n)
  e_large = rng.normal(loc = 0.0, scale = tau, size = n)
  epsilon = np.where(u < prob, e_large, e_small)
  y = b0 + b1*xi + b2*xi**2 +  epsilon
  return xi,y
'''
def fit_ols_quadratic(xi, y):
  X = np.column_stack([np.ones(len(xi)), xi, xi**2])
  fit = sm.OLS(y,X).fit()
  return np.asarray(fit.params)

def fit_lad_quadratic(xi, y):
  X = np.column_stack([np.ones(len(xi)), xi, xi**2])
  fit = sm.QuantReg(y,X).fit(q=0.5, max_iter=10000, p_tol=1e-10)
  return np.asarray(fit.params)


def quadratic_fsd(xi, y, time_limit = 120):
  S = len(y)
  # benchmark
  c_ols = fit_ols(xi,y)
  Lb = np.abs(y-(c_ols[0] + c_ols[1]*xi))
  #c_ols = fit_ols_quadratic(xi,y)
  #Lb = np.abs(y-(c_ols[0] + c_ols[1]*xi + c_ols[2]*xi**2))
  t, counts = np.unique(Lb, return_counts = True) # ordered support points
  q = counts/S # prob of each scenario for benchmark
  K = len(t)
  p = np.full(S, 1.0/S) # prob of each scenario of xi

  # Model
  m = Model(env = env)

  # bounds on c
  C = 20
  # variables
  c0 = m.addVar(lb = -C, ub = C, name = "c0")
  c1 = m.addVar(lb = -C, ub = C, name = "c1")
  c2 = m.addVar(lb = -C, ub = C, name = "c2")

  beta = m.addVars(S, K , vtype = GRB.BINARY, name = "beta")

  # linearize absolute value
  r = m.addVars(S, lb = -GRB.INFINITY, name = "r")
  u_abs = m.addVars(S, lb = 0.0, name = "u_abs")

  y_max = float(np.max(np.abs(y)))
  xi_max = float(np.max(np.abs(xi)))
  xi2_max = float(np.max(np.abs(xi**2)))

  M = np.maximum(0.0, y_max + C + C*xi_max + C*xi2_max - t) # get upper bound with triangle ineq

  # objective
  m.setObjective(1/S*quicksum(u_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - (c0 + c1*xi[s] + c2*xi[s]**2) for s in range(S))
  m.addConstrs((u_abs[s] >= r[s] for s in range(S)))
  m.addConstrs((u_abs[s] >= -r[s] for s in range(S)))

  prob_bench = 1.0 - np.cumsum(q)
  m.addConstrs(quicksum(p[s]*beta[s,k] for s in range(S)) <= prob_bench[k] for k in range(K))
  m.addConstrs(M[k]*beta[s,k] >= u_abs[s] - t[k] for k in range(K) for s in range(S))

  # Solve
  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.setParam("TimeLimit", time_limit)

  m.optimize()

  if m.status != GRB.OPTIMAL:
    print("failed. Status:", m.status)
    print("Solution count:", m.SolCount)
    return None, m.status
  return np.array([c0.X, c1.X, c2.X]), m.status

def check_fsd_sol_quadratic(xi, y, c_fsd, c_ols):

  L_fsd = np.abs(y- (c_fsd[0] + c_fsd[1]*xi + c_fsd[2]*xi**2))
  L_ols = np.abs(y - (c_ols[0] + c_ols[1]*xi))
  #L_ols = np.abs(y - (c_ols[0] + c_ols[1]*xi + c_ols[2]*xi**2))
  n = len(L_ols)

  #benchmark
  t, counts = np.unique(L_ols, return_counts = True)

  tol = 1e-6

  cdf_fsd = np.array([(L_fsd <= tk + tol).mean() for tk in t])
  cdf_ols = np.array([(L_ols <= tk).mean() for tk in t])

  gaps_cdf = cdf_fsd - cdf_ols # ols cdf should be under ols always

  min_gap = gaps_cdf.min()

  return min_gap >= -tol


In [ ]:
def run_mc_quadratic(n_rep, n = 100):
  solved = np.zeros(n_rep, dtype = bool)
  c_ols = np.zeros((n_rep,2))
  c_quad_lad = np.zeros((n_rep,3))
  c_quad_ols = np.zeros((n_rep,3))
  c_fsd = np.zeros((n_rep,3))
  mae_fsd = np.zeros((n_rep))
  mae_quad_ols = np.zeros((n_rep))
  mae_quad_lad = np.zeros((n_rep))
  mae_lin_ols = np.zeros((n_rep))

  q95_loss_fsd = np.zeros((n_rep))
  q95_loss_quad_lad = np.zeros((n_rep))
  q95_loss_quad_ols = np.zeros((n_rep))
  q95_loss_ols = np.zeros((n_rep))

  q99_loss_fsd = np.zeros((n_rep))
  q99_loss_quad_lad = np.zeros((n_rep))
  q99_loss_quad_ols = np.zeros((n_rep))
  q99_loss_ols = np.zeros((n_rep))

  quad_olsVSfsd_mae = np.zeros((n_rep))
  quad_olsVSfsd_95 = np.zeros((n_rep))
  quad_olsVSfsd_99 = np.zeros((n_rep))

  quad_ladVSfsd_mae = np.zeros((n_rep))
  quad_ladVSfsd_95 = np.zeros((n_rep))
  quad_ladVSfsd_99 = np.zeros((n_rep))

  coef_close_ols = np.zeros(n_rep)
  coef_close_lad = np.zeros(n_rep)

  for i in range(n_rep):
    if (i % 10 == 0):
      print("iteration:", i)
    seed = 542 + i # for reproducibility
    xi, y = simulate_data_quadratic(n, seed = seed)

    sol,status = quadratic_fsd(xi, y)
    if (status != GRB.OPTIMAL):
      print("FSD not solved at iteration", i, "status:", status)
      continue # if not solved optimally go to next dataset
    c_fsd[i] = sol
    c_quad_lad[i] = fit_lad_quadratic(xi,y)
    c_ols[i] = fit_ols(xi, y)
    c_quad_ols[i] = fit_ols_quadratic(xi, y)

    mae_fsd[i] = np.mean(abs(y - (c_fsd[i,0] + c_fsd[i,1] * xi + c_fsd[i,2]*xi**2 )))
    mae_lin_ols[i] = np.mean(abs(y - (c_ols[i,0] + c_ols[i,1] * xi)))
    mae_quad_lad[i] = np.mean(abs(y - (c_quad_lad[i,0] + c_quad_lad[i,1] * xi + c_quad_lad[i,2]*xi**2)))
    mae_quad_ols[i] = np.mean(abs(y - (c_quad_ols[i,0] + c_quad_ols[i,1] * xi + c_quad_ols[i,2]*xi**2)))

    if (check_fsd_sol_quadratic(xi, y, c_fsd[i], c_ols[i] )):
      solved[i] = True
      if(mae_fsd[i] < mae_quad_ols[i] - 1e-6):
        quad_olsVSfsd_mae[i] = 1
      if(mae_fsd[i]< mae_quad_lad[i] -1e-6):
        quad_ladVSfsd_mae[i] = 1

      L_lin_ols = np.abs(y - (c_ols[i,0] + c_ols[i,1]*xi))
      L_quad_ols = np.abs(y - (c_quad_ols[i,0] + c_quad_ols[i,1]*xi + c_quad_ols[i,2]*xi**2))
      L_quad_lad = np.abs(y - (c_quad_lad[i,0] + c_quad_lad[i,1]*xi + c_quad_lad[i,2]*xi**2))
      L_fsd = np.abs(y - (c_fsd[i,0] + c_fsd[i,1]*xi + c_fsd[i,2]*xi**2))

      q95_loss_fsd[i] = float(np.quantile(L_fsd, 0.95))
      q95_loss_quad_ols[i] = float(np.quantile(L_quad_ols, 0.95))
      q95_loss_quad_lad[i] = float(np.quantile(L_quad_lad, 0.95))
      q95_loss_ols[i] = float(np.quantile(L_lin_ols, 0.95))

      q99_loss_fsd[i] = float(np.quantile(L_fsd, 0.99))
      q99_loss_quad_ols[i] = float(np.quantile(L_quad_ols, 0.99))
      q99_loss_quad_lad[i] = float(np.quantile(L_quad_lad, 0.99))
      q99_loss_ols[i] = float(np.quantile(L_lin_ols, 0.99))

      if(q95_loss_fsd[i] <q95_loss_quad_ols[i]-1e-6):
        quad_olsVSfsd_95[i] = 1
      if(q95_loss_fsd[i]< q95_loss_quad_lad[i] -1e-6):
        quad_ladVSfsd_95[i] = 1

      if(q99_loss_fsd[i] < q99_loss_quad_ols[i]-1e-6):
        quad_olsVSfsd_99[i] = 1
      if(q99_loss_fsd[i] < q99_loss_quad_lad[i]-1e-6):
        quad_ladVSfsd_99[i] = 1

      if (np.linalg.norm(c_fsd[i] - c_quad_ols[i], ord=np.inf) <= 1e-6):
        coef_close_ols[i] = 1

      if (np.linalg.norm(c_fsd[i] - c_quad_lad[i], ord=np.inf) <= 1e-6):
        coef_close_lad[i] = 1

    else:
      print("fsd failed at iteration", i)

  print("Number of solved and valid instances", solved.sum())

  print("Average MAE (OLS): ", mae_lin_ols[solved].mean()) # only calculate solved instances
  print("Average MAE (quadratic LAD): ", mae_quad_lad[solved].mean())
  print("Average MAE (FSD): ", mae_fsd[solved].mean())
  print("Average MAE (quadratic OLS): ", mae_quad_ols[solved].mean())

  print("Average q95 loss (linear OLS):", q95_loss_ols[solved].mean())
  print("Average q95 loss (quadratic OLS):", q95_loss_quad_ols[solved].mean())
  print("Average q95 loss (quadratic LAD):", q95_loss_quad_lad[solved].mean())
  print("Average q95 loss (FSD):", q95_loss_fsd[solved].mean())

  print("Average q99 loss (linear OLS):", q99_loss_ols[solved].mean())
  print("Average q99 loss (quadratic OLS):", q99_loss_quad_ols[solved].mean())
  print("Average q99 loss (quadratic LAD):", q99_loss_quad_lad[solved].mean())
  print("Average q99 loss (FSD):", q99_loss_fsd[solved].mean())

  print("Fraction where FSD beats quadratic OLS in MAE", quad_olsVSfsd_mae[solved].mean())
  print("Fraction where FSD beats quadratic OLS in q95", quad_olsVSfsd_95[solved].mean())
  print("Fraction where FSD beats quadratic OLS in q99", quad_olsVSfsd_99[solved].mean())
  print("Fraction where FSD beats quadratic LAD in MAE", quad_ladVSfsd_mae[solved].mean())
  print("Fraction where FSD beats quadratic LAD in q95", quad_ladVSfsd_95[solved].mean())
  print("Fraction where FSD beats quadratic LAD in q99", quad_ladVSfsd_99[solved].mean())

  print("Fraction where FSD = quadratic OLS (tol 10^(-6))", coef_close_ols[solved].mean())
  print("Fraction where FSD = quadratic LAD (tol 10^(-6))", coef_close_lad[solved].mean())

run_mc_quadratic(n_rep = 500, n = 100)

SSD

Start with simple linear regression model with same error terms as in FSD

In [ ]:
def model_SSD(xi, y):
  S = len(y)
  c_ols = fit_ols(xi,y)
  Lb = np.abs(y - (c_ols[0] + c_ols[1]*xi))

  t, counts = np.unique(Lb, return_counts = True) # ordered support points
  q = counts/S # prob of each scenario for benchmark
  K = len(t)

  p = np.full(S, 1/S) # prob of each scenario of xi

  # Model
  m = Model(env = env)

  # bounds on c
  C = 20
  # variables
  c0 = m.addVar(lb = -C, ub = C, name = "c0")
  c1 = m.addVar(lb = -C, ub = C, name = "c1")

  # linearize absolute value
  r = m.addVars(S, lb = -GRB.INFINITY, name = "r")
  l_abs = m.addVars(S, lb = 0.0, name = "l_abs")
  u = m.addVars(S,K, lb = 0.0, name = "u")

  # objective
  m.setObjective(1/S*quicksum(l_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - (c0 + c1*xi[s]) for s in range(S))
  m.addConstrs((l_abs[s] >= r[s] for s in range(S)))
  m.addConstrs((l_abs[s] >= -r[s] for s in range(S)))

  rhs_ssd = np.array([np.sum(q*np.maximum(t-t[k], 0.0)) for k in range(K)])
  m.addConstrs(quicksum(p[s]*u[s,k] for s in range(S)) <=rhs_ssd[k] for k in range(K))
  m.addConstrs(u[s,k] >= l_abs[s] - t[k] for k in range(K) for s in range(S))

  # Solve
  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.optimize()
  return c0.X, c1.X


SSD Monte Carlo - linear model

In [ ]:
def run_mc_ssd(n_rep , n):
  mean_ssd = np.zeros((n_rep, 2))
  c_ols = np.zeros((n_rep,2))
  c_lad = np.zeros((n_rep,2))
  c_ssd = np.zeros((n_rep,2))
  mae_ssd = np.zeros((n_rep))
  mae_ols = np.zeros((n_rep))
  mae_lad = np.zeros((n_rep))
  coef_close = np.zeros(n_rep)

  for i in range(n_rep):
    if (i % 50 == 0):
      print("iteration:", i)
    seed = 1234 + i # for reproducibility
    #xi, y = simulate_data(n, seed = seed)
    xi, y = simulate_asymmetric(n, seed = seed)
    c_ssd[i] = model_SSD(xi, y)
    c_lad[i] = fit_lad(xi,y)
    c_ols[i] = fit_ols(xi, y)

    mae_ssd[i] = np.mean(abs(y - (c_ssd[i,0] + c_ssd[i,1] * xi)))
    mae_ols[i] = np.mean(abs(y - (c_ols[i,0] + c_ols[i,1] * xi)))
    mae_lad[i] = np.mean(abs(y - (c_lad[i,0] + c_lad[i,1] * xi)))

    if (np.linalg.norm(c_ssd[i] - c_ols[i], ord=np.inf) <= 1e-7):
      coef_close[i] = 1
    else:
      print("ols", c_ols[i])
      print("ssd", c_ssd[i])

  print("Average MAE (OLS): ", mae_ols.mean())
  print("Average MAE (LAD): ", mae_lad.mean())
  print("Average MAE (SSD): ", mae_ssd.mean())
  print("Average coef (SSD): ", c_ssd.mean(axis=0))
  print("Average coef (OLS): ", c_ols.mean(axis = 0))
  print("Average coef (LAD): ", c_lad.mean(axis = 0))
  print("Fraction where OLS = SSD", coef_close.mean())

run_mc_ssd(n_rep = 500, n = 100)

SSD Multidimensional

In [ ]:
def ssd_multi(xi,y):
  S, d = xi.shape
  X = sm.add_constant(xi)
  c_ols = fit_ols(xi,y)

  Lb = np.abs(y - X@c_ols)
  t, counts = np.unique(Lb, return_counts = True)
  q = counts/S
  K = len(t)
  p = np.full(S, 1/S)

  # Model
  m = Model(env = env)
  # upper bound
  C = 20

  # variables
  # c[0] intercept and c[1],..,c[d] slopes
  c = m.addVars(d+1, lb = -C, ub = C, name = "c")
  r = m.addVars(S,lb = -GRB.INFINITY, name = "r")
  l_abs = m.addVars(S, lb = 0.0, name = "l_abs")
  u = m.addVars(S, K, lb = 0.0, name = "u")

  # objective
  m.setObjective(1/S*quicksum(l_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - quicksum(c[j]*X[s,j] for j in range(d+1)) for s in range(S))
  m.addConstrs(l_abs[s] >= r[s] for s in range(S))
  m.addConstrs(l_abs[s] >= -r[s] for s in range(S))

  rhs_ssd = np.array([np.sum(q*np.maximum(t-t[k], 0.0)) for k in range(K)])
  m.addConstrs(quicksum(p[s]*u[s,k] for s in range(S)) <= rhs_ssd[k] for k in range(K))
  m.addConstrs(u[s,k] >= l_abs[s] - t[k] for k in range(K) for s in range(S))

  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)
  m.optimize()

  return np.array([c[j].X for j in range(d + 1)])

In [ ]:
def run_mc_multid_ssd(dim, beta, n_rep = 100, n = 100):
  mean_ssd = np.zeros((n_rep, dim+1))
  c_ols = np.zeros((n_rep,dim+1))
  c_lad = np.zeros((n_rep, dim + 1))
  c_ssd = np.zeros((n_rep, dim + 1))
  coef_close = np.zeros(n_rep)
  for i in range(n_rep):
    if (i % 50 == 0):
      print("iteration:", i)
    seed = 1234 + i # for reproducibility
    xi, y = simulate_multid_data(n, dim = dim, beta = beta, seed = seed) # or asymetric data
    c_ssd[i] = ssd_multi(xi, y)
    c_lad[i] = fit_lad(xi,y)
    c_ols[i] = fit_ols(xi, y)

    if (np.linalg.norm(c_ssd[i] - c_ols[i], ord=np.inf) <= 1e-7):
      coef_close[i] = 1
    else:
      print("ols", c_ols[i])
      print("ssd", c_ssd[i])


  print("Average coef (SSD): ", c_ssd.mean(axis=0))
  print("Average coef (OLS): ", c_ols.mean(axis = 0))
  print("Average coef (LAD): ", c_lad.mean(axis = 0))
  print("Fraction where OLS = SSD", coef_close.mean())


run_mc_multid_ssd(2, beta = [2.0, 3.0], n_rep = 100, n = 100)

SSD Piecewise linear model

In [ ]:
def piecewise_ssd(xi, y, kappa = 0.0):
  S = len(y)
  z = np.maximum(xi - kappa, 0.0)
  # benchmark
  c_ols = fit_ols(xi,y)
  Lb = np.abs(y-(c_ols[0] + c_ols[1]*xi))
  t, counts = np.unique(Lb, return_counts = True) # ordered support points
  q = counts/S # prob of each scenario for benchmark
  K = len(t)
  p = np.full(S, 1.0/S) # prob of each scenario of xi

  # Model
  m = Model(env = env)

  # bounds on c
  C = 20
  # variables
  c0 = m.addVar(lb = -C, ub = C, name = "c0")
  c1 = m.addVar(lb = -C, ub = C, name = "c1")
  c2 = m.addVar(lb = -C, ub = C, name = "c2")

  # linearize absolute value
  r = m.addVars(S, lb = -GRB.INFINITY, name = "r")
  l_abs = m.addVars(S, lb = 0.0, name = "l_abs")
  u = m.addVars(S, K, lb=0.0, name = "u")

  # objective
  m.setObjective(1/S*quicksum(l_abs[s] for s in range(S)), GRB.MINIMIZE)

  # constraints
  m.addConstrs(r[s] == y[s] - (c0 + c1*xi[s] + c2*z[s]) for s in range(S))
  m.addConstrs((l_abs[s] >= r[s] for s in range(S)))
  m.addConstrs((l_abs[s] >= -r[s] for s in range(S)))

  rhs_ssd = np.array([np.sum(q*np.maximum(t-t[k], 0.0)) for k in range(K)])
  m.addConstrs(quicksum(p[s]*u[s,k] for s in range(S)) <= rhs_ssd[k] for k in range(K))
  m.addConstrs(u[s,k] >= l_abs[s] - t[k] for k in range(K) for s in range(S))

  # Solve
  m.setParam("OutputFlag", 0)
  m.setParam("FeasibilityTol", 1e-9)
  m.setParam("IntFeasTol", 1e-8)

  m.optimize()

  if m.status != GRB.OPTIMAL:
    #print("failed. Status:", m.status)
    #print("Solution count:", m.SolCount)
    return None, m.status
  return np.array([c0.X, c1.X, c2.X]), m.status

In [ ]:
def run_mc_piecewise_ssd(n_rep, n = 100, kappa = 0.0):

  c_ols = np.zeros((n_rep,2))
  c_piece_lad = np.zeros((n_rep,3))
  c_piece_ols = np.zeros((n_rep,3))
  c_ssd = np.zeros((n_rep,3))
  mae_ssd = np.zeros((n_rep))
  mae_piece_ols = np.zeros((n_rep))
  mae_piece_lad = np.zeros((n_rep))
  mae_lin_ols = np.zeros((n_rep))

  q95_loss_ssd = np.zeros((n_rep))
  q95_loss_piece_lad = np.zeros((n_rep))
  q95_loss_piece_ols = np.zeros((n_rep))
  q95_loss_ols = np.zeros((n_rep))

  q99_loss_ssd = np.zeros((n_rep))
  q99_loss_piece_lad = np.zeros((n_rep))
  q99_loss_piece_ols = np.zeros((n_rep))
  q99_loss_ols = np.zeros((n_rep))

  piece_olsVSssd_mae = np.zeros((n_rep))
  piece_olsVSssd_95 = np.zeros((n_rep))
  piece_olsVSssd_99 = np.zeros((n_rep))

  piece_ladVSssd_mae = np.zeros((n_rep))
  piece_ladVSssd_95 = np.zeros((n_rep))
  piece_ladVSssd_99 = np.zeros((n_rep))

  coef_close_ols = np.zeros(n_rep)
  coef_close_lad = np.zeros(n_rep)

  for i in range(n_rep):
    if (i % 50 == 0):
      print("iteration:", i)
    seed = 542 + i # for reproducibility
    xi, y = simulate_data_piecewise(n, seed = seed, kappa = kappa)
    z = np.maximum(xi - kappa, 0.0)

    sol,status = piecewise_ssd(xi, y, kappa = kappa)
    if (status != GRB.OPTIMAL):
      print("SSD not solved at iteration", i, "status:", status)
      continue # if not solved optimally go to next dataset
    c_ssd[i] = sol
    c_piece_lad[i] = fit_lad_piecewise(xi,y, kappa = kappa)
    c_ols[i] = fit_ols(xi, y)
    c_piece_ols[i] = fit_ols_piecewise(xi, y, kappa = kappa)

    mae_ssd[i] = np.mean(abs(y - (c_ssd[i,0] + c_ssd[i,1] * xi + c_ssd[i,2]*z )))
    mae_lin_ols[i] = np.mean(abs(y - (c_ols[i,0] + c_ols[i,1] * xi)))
    mae_piece_lad[i] = np.mean(abs(y - (c_piece_lad[i,0] + c_piece_lad[i,1] * xi + c_piece_lad[i,2]*z)))
    mae_piece_ols[i] = np.mean(abs(y - (c_piece_ols[i,0] + c_piece_ols[i,1] * xi + c_piece_ols[i,2]*z)))

    if(mae_ssd[i] < mae_piece_ols[i] - 1e-6):
      piece_olsVSssd_mae[i] = 1
    if(mae_ssd[i]< mae_piece_lad[i] -1e-6):
      piece_ladVSssd_mae[i] = 1

    L_lin_ols = np.abs(y - (c_ols[i,0] + c_ols[i,1]*xi))
    L_piece_ols = np.abs(y - (c_piece_ols[i,0] + c_piece_ols[i,1]*xi + c_piece_ols[i,2]*z))
    L_piece_lad = np.abs(y - (c_piece_lad[i,0] + c_piece_lad[i,1]*xi + c_piece_lad[i,2]*z))
    L_ssd = np.abs(y - (c_ssd[i,0] + c_ssd[i,1]*xi + c_ssd[i,2]*z))

    q95_loss_ssd[i] = float(np.quantile(L_ssd, 0.95))
    q95_loss_piece_ols[i] = float(np.quantile(L_piece_ols, 0.95))
    q95_loss_piece_lad[i] = float(np.quantile(L_piece_lad, 0.95))
    q95_loss_ols[i] = float(np.quantile(L_lin_ols, 0.95))

    q99_loss_ssd[i] = float(np.quantile(L_ssd, 0.99))
    q99_loss_piece_ols[i] = float(np.quantile(L_piece_ols, 0.99))
    q99_loss_piece_lad[i] = float(np.quantile(L_piece_lad, 0.99))
    q99_loss_ols[i] = float(np.quantile(L_lin_ols, 0.99))

    if(q95_loss_ssd[i] <q95_loss_piece_ols[i]-1e-6):
      piece_olsVSssd_95[i] = 1
    if(q95_loss_ssd[i]< q95_loss_piece_lad[i] -1e-6):
      piece_ladVSssd_95[i] = 1

    if(q99_loss_ssd[i] < q99_loss_piece_ols[i]-1e-6):
      piece_olsVSssd_99[i] = 1
    if(q99_loss_ssd[i] < q99_loss_piece_lad[i]-1e-6):
      piece_ladVSssd_99[i] = 1

    if (np.linalg.norm(c_ssd[i] - c_piece_ols[i], ord=np.inf) <= 1e-6):
      coef_close_ols[i] = 1
    if (np.linalg.norm(c_ssd[i] - c_piece_lad[i], ord=np.inf) <= 1e-6):
      coef_close_lad[i] = 1


  print("Average MAE (OLS): ", mae_lin_ols.mean()) # only calculate solved instances
  print("Average MAE (piecewise LAD): ", mae_piece_lad.mean())
  print("Average MAE (SSD): ", mae_ssd.mean())
  print("Average MAE (piecewise OLS): ", mae_piece_ols.mean())

  print("Average q95 loss (linear OLS):", q95_loss_ols.mean())
  print("Average q95 loss (piecewise OLS):", q95_loss_piece_ols.mean())
  print("Average q95 loss (piecewise LAD):", q95_loss_piece_lad.mean())
  print("Average q95 loss (sSD):", q95_loss_ssd.mean())

  print("Average q99 loss (linear OLS):", q99_loss_ols.mean())
  print("Average q99 loss (piecewise OLS):", q99_loss_piece_ols.mean())
  print("Average q99 loss (piecewise LAD):", q99_loss_piece_lad.mean())
  print("Average q99 loss (SSD):", q99_loss_ssd.mean())

  print("Fraction where SSD beats piecewise OLS in MAE", f"{piece_olsVSssd_mae.mean():.2f}")
  print("Fraction where SSD beats piecewise OLS in q95", f"{piece_olsVSssd_95.mean():.2f}")
  print("Fraction where SSD beats piecewise OLS in q99", f"{piece_olsVSssd_99.mean():.2f}")
  print("Fraction where SSD beats piecewise LAD in MAE", f"{piece_ladVSssd_mae.mean():.2f}")
  print("Fraction where SSD beats piecewise LAD in q95", f"{piece_ladVSssd_95.mean():.2f}")
  print("Fraction where SSD beats piecewise LAD in q99", f"{piece_ladVSssd_99.mean():.2f}")

  print("Fraction where SSD = piecewise OLS (tol 10^(-6))", coef_close_ols.mean())
  print("Fraction where SSD = piecewise LAD (tol 10^(-6))", coef_close_lad.mean())


run_mc_piecewise_ssd(n_rep = 500, n = 100, kappa = 0.0)